# SAP → Snowflake generator

From a list of SAP tables, generate everything to land + model them in Snowflake/dbt
from the SAP data dictionary (DD02L/DD02T/DD03L/DD04T). Per table:

1. **LZ / bronze / silver DDLs** — typed `CREATE TABLE`s as snowfiles (`USE ROLE` + env tokens), **audit columns on top**
2. **dbt bronze model** — `source()`, technical names, audit cols, *optional soft-delete capture* (`IS_DELETED`)
3. **dbt silver model** — `ref(bronze)`, business names + max-type casts (carries `IS_DELETED` when delete capture is on)
4. **dbt source yaml** — `_src/<source>.yml` (upserted)
5. **table-config entry** — under the `extraction` key of `TelTools_SapR3_<config_prefix>.json`

Then it can **run the `getSapR3Data` extract**. Logic lives in **`sap_to_snowflake.py`**; edit the **one config cell**.

## 0 · Config (everything lives here)

In [1]:
from pathlib import Path
import sap_to_snowflake as s

cfg = s.SapModelConfig(
    connection = dict(
        account="telus-global.privatelink", user="stanley.x.correia@us.tel.com",
        authenticator="externalbrowser", warehouse="DEVELOPER_WH", role="SYSADMIN_DEV",
    ),

    # --- snowflake locations ---
    lz_database="LZ_DEV", dd_schema="SAP_ORACLE_LIVE",
    source_name="SAP_ORACLE_LIVE", snowflake_schema="SAP_ORACLE_LIVE", sap_schema="SAPR3",

    # --- dbt / repo ---
    repo_root=Path(r"C:\EnterpriseCloud"), dbt_project="dbt_tel",
    dbt_subfolder="TMEA_FINANCE", source_prefix="SAPO",

    # --- bronze / silver physical tables (default to landing if omitted) ---
    bronze_db="BRONZE_DEV", bronze_schema="TMEA_FINANCE", bronze_role="PLATFORM_TEAM_DEV",
    silver_db="SILVER_DEV", silver_schema="TMEA_FINANCE", silver_role="PLATFORM_TEAM_DEV",

    # --- behavior ---
    incremental_strategy="insert_overwrite", capture_deletes=False,
    chunk_size=30000, silver_numeric_scale=2,

    # --- table-config file: TelTools_SapR3_<config_prefix>.json, entries under <config_root_key> ---
    config_prefix="InsertOverwrite", config_root_key="extraction",

    # --- snowfile (roles + env tokens); one ticket, three seqs/files ---
    snow_ticket="K.42787", lz_seq=1, bronze_seq=2, silver_seq=3,
    ingestion_label="Ingest_Tmea_Finance_Tables", lz_role="SYSADMIN_DEV",

    # --- extract runner ---
    extract_env="PROD", extract_debug=False, extract_chunk_size=300000,

    # --- tables: "NAME" OR {"table","friendly","strategy","capture_deletes"} ---
    tables = [
        # {"table": "STAS", "friendly": "BOMS_ITEM_SELECTION", "capture_deletes": True},
        # {"table": "CDPOS", "strategy": "merge"},
        # "DD02T",
        {"table": "SKA1", "strategy": "merge"},
        {"table": "GLT0", "strategy": "merge"},
        # "SKA1",
        # "GLT0"
    ],
)
for spec in cfg.tables:
    t = s.normalize_table_spec(spec, cfg)
    print(f"{t['table']:<8} strategy={t['strategy']:<16} deletes={t['capture_deletes']}")
print("\nsnowfiles:", cfg.lz_snow_file.name, "|", cfg.bronze_snow_file.name, "|", cfg.silver_snow_file.name)

SKA1     strategy=merge            deletes=False
GLT0     strategy=merge            deletes=False

snowfiles: K.42787__1.lz_Ingest_Tmea_Finance_Tables__data.SQL | K.42787__2.bronze_Ingest_Tmea_Finance_Tables__data.SQL | K.42787__3.silver_Ingest_Tmea_Finance_Tables__data.SQL


## 1 · Connect & inspect one table

In [2]:
conn = s.connect(cfg.connection)
probe = s.normalize_table_spec(cfg.tables[0], cfg)
rows   = s.fetch_metadata(conn, cfg, probe["table"])
fields = s.build_fields(rows)
friendly = s.friendly_name(rows, probe["table"], probe["friendly"])
print(f"table {probe['table']} -> {friendly}  ({len(fields)} fields)")
for f in fields:
    print(f"  {'PK' if f['pk'] else '  '} {f['technical']:<15} {f['type']:<6} -> {f['business']}")

table SKA1 -> G_L_ACCOUNT_MASTER_CHART_OF_ACCOUNTS  (18 fields)
  PK MANDT           CLNT   -> CLIENT
  PK KTOPL           CHAR   -> CHART_OF_ACCOUNTS
  PK SAKNR           CHAR   -> G_L_ACCOUNT_NUMBER
     XBILK           CHAR   -> INDICATOR_ACCOUNT_IS_A_BALANCE_SHEET_ACCOUNT
     SAKAN           CHAR   -> G_L_ACCOUNT_NUMBER_SIGNIFICANT_LENGTH
     BILKT           CHAR   -> GROUP_ACCOUNT_NUMBER
     ERDAT           DATS   -> DATE_ON_WHICH_THE_RECORD_WAS_CREATED
     ERNAM           CHAR   -> NAME_OF_PERSON_WHO_CREATED_THE_OBJECT
     GVTYP           CHAR   -> P_L_STATEMENT_ACCOUNT_TYPE
     KTOKS           CHAR   -> G_L_ACCOUNT_GROUP
     MUSTR           CHAR   -> NUMBER_OF_THE_SAMPLE_ACCOUNT
     VBUND           CHAR   -> COMPANY_ID_OF_TRADING_PARTNER
     XLOEV           CHAR   -> INDICATOR_ACCOUNT_MARKED_FOR_DELETION
     XSPEA           CHAR   -> INDICATOR_ACCOUNT_IS_BLOCKED_FOR_CREATION
     XSPEB           CHAR   -> INDICATOR_IS_ACCOUNT_BLOCKED_FOR_POSTING
     XSPEP           CH

## 2 · Preview the models (no write)

In [3]:
print(s.build_bronze_model(cfg, fields, probe["table"], friendly, probe["strategy"],
                          capture_deletes=probe["capture_deletes"]))

{{ config(
    materialized='tel_incremental',
    incremental_strategy='merge',
    alias='SAPO_G_L_ACCOUNT_MASTER_CHART_OF_ACCOUNTS',
    unique_key = ['MANDT','KTOPL','SAKNR'],
    transient=false
) }}

with source_data as (
    SELECT
        _CAPTUREDATETIME,
        MANDT,
        KTOPL,
        SAKNR,
        XBILK,
        SAKAN,
        BILKT,
        ERDAT,
        ERNAM,
        GVTYP,
        KTOKS,
        MUSTR,
        VBUND,
        XLOEV,
        XSPEA,
        XSPEB,
        XSPEP,
        MCOD1,
        FUNC_AREA
    FROM {{ source('SAP_ORACLE_LIVE', 'SKA1') }}
),

transform_data as (
    SELECT
        '{{ invocation_id }}' AS _AUDIT_LOG_ID,
        SHA2('hash_me') AS _ROW_HASH,
        _CAPTUREDATETIME,
        MANDT AS MANDT,
        KTOPL AS KTOPL,
        SAKNR AS SAKNR,
        XBILK AS XBILK,
        SAKAN AS SAKAN,
        BILKT AS BILKT,
        ERDAT AS ERDAT,
        ERNAM AS ERNAM,
        GVTYP AS GVTYP,
        KTOKS AS KTOKS,
        MUSTR AS MUSTR,
  

In [4]:
print(s.build_silver_model(cfg, fields, friendly, probe["strategy"],
                          capture_deletes=probe["capture_deletes"]))

{{ config(
    materialized='tel_incremental',
    incremental_strategy='merge',
    alias='G_L_ACCOUNT_MASTER_CHART_OF_ACCOUNTS',
    unique_key = ['CLIENT','CHART_OF_ACCOUNTS','G_L_ACCOUNT_NUMBER'],
    transient=false
) }}

with source_data as (
    SELECT
        _CAPTUREDATETIME,
        MANDT AS CLIENT,
        KTOPL AS CHART_OF_ACCOUNTS,
        SAKNR AS G_L_ACCOUNT_NUMBER,
        XBILK AS INDICATOR_ACCOUNT_IS_A_BALANCE_SHEET_ACCOUNT,
        SAKAN AS G_L_ACCOUNT_NUMBER_SIGNIFICANT_LENGTH,
        BILKT AS GROUP_ACCOUNT_NUMBER,
        ERDAT AS DATE_ON_WHICH_THE_RECORD_WAS_CREATED,
        ERNAM AS NAME_OF_PERSON_WHO_CREATED_THE_OBJECT,
        GVTYP AS P_L_STATEMENT_ACCOUNT_TYPE,
        KTOKS AS G_L_ACCOUNT_GROUP,
        MUSTR AS NUMBER_OF_THE_SAMPLE_ACCOUNT,
        VBUND AS COMPANY_ID_OF_TRADING_PARTNER,
        XLOEV AS INDICATOR_ACCOUNT_MARKED_FOR_DELETION,
        XSPEA AS INDICATOR_ACCOUNT_IS_BLOCKED_FOR_CREATION,
        XSPEB AS INDICATOR_IS_ACCOUNT_BLOCKED_FOR_POSTI

## 3 · Preview the DDLs (audit columns on top)

In [5]:
print(s.build_lz_ddl(cfg, fields, probe["table"]))
print()
print(s.build_bronze_ddl(cfg, fields, friendly, capture_deletes=probe["capture_deletes"]))
print()
print(s.build_silver_ddl(cfg, fields, friendly, capture_deletes=probe["capture_deletes"]))

create or replace TABLE SKA1 (
	MANDT VARCHAR(9) NOT NULL DEFAULT '',
	KTOPL VARCHAR(12) NOT NULL DEFAULT '',
	SAKNR VARCHAR(30) NOT NULL DEFAULT '',
	XBILK VARCHAR(3) NOT NULL DEFAULT '',
	SAKAN VARCHAR(30) NOT NULL DEFAULT '',
	BILKT VARCHAR(30) NOT NULL DEFAULT '',
	ERDAT VARCHAR(24) NOT NULL DEFAULT '',
	ERNAM VARCHAR(36) NOT NULL DEFAULT '',
	GVTYP VARCHAR(6) NOT NULL DEFAULT '',
	KTOKS VARCHAR(12) NOT NULL DEFAULT '',
	MUSTR VARCHAR(30) NOT NULL DEFAULT '',
	VBUND VARCHAR(18) NOT NULL DEFAULT '',
	XLOEV VARCHAR(3) NOT NULL DEFAULT '',
	XSPEA VARCHAR(3) NOT NULL DEFAULT '',
	XSPEB VARCHAR(3) NOT NULL DEFAULT '',
	XSPEP VARCHAR(3) NOT NULL DEFAULT '',
	MCOD1 VARCHAR(75) NOT NULL DEFAULT '',
	FUNC_AREA VARCHAR(48) NOT NULL DEFAULT '',
	_CAPTUREDATETIME TIMESTAMP_LTZ(9) NOT NULL DEFAULT CAST('0001-01-01 00:00:00' AS TIMESTAMP_LTZ(9)),
	_FILENAME VARCHAR(16777216),
	_FILEDATE VARCHAR(16777216),
	_UPLOADDATE VARCHAR(16777216),
	primary key (MANDT, KTOPL, SAKNR)
);

CREATE OR REPLACE TA

## 4 · Generate everything
Writes bronze + silver models, source YAML, the three DDL snowfiles (LZ / bronze / silver),
and upserts each table under the `extraction` key of the config.

In [6]:
results = s.generate_all(cfg, conn=conn, write_ddl=True, write_source=True)
for k, v in results.items():
    if k.startswith("_"):
        print(f"{k}: {v}")
    else:
        print(f"{k} ({v['friendly']})\n   bronze: {v['bronze']}\n   silver: {v['silver']}")

  -> SKA1 (merge)
  -> GLT0 (merge)
SKA1 (G_L_ACCOUNT_MASTER_CHART_OF_ACCOUNTS)
   bronze: C:\EnterpriseCloud\astro_datautility\dags\dbt\dbt_tel\models\tmea_finance\bronze\BRONZE__TMEA_FINANCE__SAPO_G_L_ACCOUNT_MASTER_CHART_OF_ACCOUNTS.sql
   silver: C:\EnterpriseCloud\astro_datautility\dags\dbt\dbt_tel\models\tmea_finance\silver\SILVER__TMEA_FINANCE__G_L_ACCOUNT_MASTER_CHART_OF_ACCOUNTS.sql
GLT0 (G_L_ACCOUNT_MASTER_RECORD_TRANSACTION_FIGURES)
   bronze: C:\EnterpriseCloud\astro_datautility\dags\dbt\dbt_tel\models\tmea_finance\bronze\BRONZE__TMEA_FINANCE__SAPO_G_L_ACCOUNT_MASTER_RECORD_TRANSACTION_FIGURES.sql
   silver: C:\EnterpriseCloud\astro_datautility\dags\dbt\dbt_tel\models\tmea_finance\silver\SILVER__TMEA_FINANCE__G_L_ACCOUNT_MASTER_RECORD_TRANSACTION_FIGURES.sql
_table_config: C:\EnterpriseCloud\astro_datautility\include\Configuration\TelTools_SapR3_InsertOverwrite.json
_source_yaml: C:\EnterpriseCloud\astro_datautility\dags\dbt\dbt_tel\models\_src\sap_oracle_live.yml
_lz_snowf

## 5 · Run the extract (optional)

In [7]:
s.run_extracts(cfg, dry_run=True)   # prints the getSapR3Data command per table

$ python -m include.Python.getSapR3Data --config-file C:\EnterpriseCloud\astro_datautility\include\Configuration\TelTools_SapR3_InsertOverwrite.json --extraction-name SKA1 --env PROD --chunk-size 300000 --debug False
$ python -m include.Python.getSapR3Data --config-file C:\EnterpriseCloud\astro_datautility\include\Configuration\TelTools_SapR3_InsertOverwrite.json --extraction-name GLT0 --env PROD --chunk-size 300000 --debug False

(dry run — call run_extracts(cfg, dry_run=False) to execute)


[]

In [7]:
s.run_extracts(cfg, dry_run=False)   # uncomment to actually run

$ python -m include.Python.getSapR3Data --config-file C:\EnterpriseCloud\astro_datautility\include\Configuration\TelTools_SapR3_InsertOverwrite.json --extraction-name SKA1 --env PROD --chunk-size 300000 --debug False
  return code: 0
-> Establishing Oracle connection to ERP for environment: PROD
   - Using host: 172.26.95.63
  STRATEGY   : INSERT OVERWRITE (via TEMP table)
  SOURCE     : SAPR3.SKA1
  TEMP TABLE : LZ_DEV.SAP_ORACLE_LIVE.SKA1__0775919F__TEMP
  MAIN TABLE : LZ_DEV.SAP_ORACLE_LIVE.SKA1
  TOTAL ROWS : 38,112
  CHUNK SIZE : 300,000
  QUEUE SIZE : 5
  TEMP CLEANUP | No stale TEMP tables found for SKA1
  TEMP TABLE | Creating : LZ_DEV.SAP_ORACLE_LIVE.SKA1__0775919F__TEMP
  TEMP TABLE | Created successfully.

  COLD START from Oracle | TIME: 0.04s
  ORACLE -> QUEUE | Chunk: 1 | Rows staged: 38,112 | Oracle fetch: 13.07s | Queue put: 0.00s | Queue depth: 1

Chunk 1 uploaded 
[1] success | True
[2] num_chunks | 1
[3] num_rows | 38112
[4] snowflake | Uploaded to LZ_DEV.SAP_ORACLE_

[('python -m include.Python.getSapR3Data --config-file C:\\EnterpriseCloud\\astro_datautility\\include\\Configuration\\TelTools_SapR3_InsertOverwrite.json --extraction-name SKA1 --env PROD --chunk-size 300000 --debug False',
  0),
 ('python -m include.Python.getSapR3Data --config-file C:\\EnterpriseCloud\\astro_datautility\\include\\Configuration\\TelTools_SapR3_InsertOverwrite.json --extraction-name GLT0 --env PROD --chunk-size 300000 --debug False',
  0)]

In [ ]:
conn.close()
print("done")

## Notes / assumptions
- **Audit columns** (`_AUDIT_LOG_ID`, `_ROW_HASH`, `_CAPTUREDATETIME`) sit at the **top** of every
  model CTE and every DDL.
- **DDLs**: LZ + bronze use precise `LENG`-based types; silver uses max types (`VARCHAR`,
  `NUMBER(38,<scale>)`, `DATE`, `TIME`). Each is a snowfile with `USE ROLE` + env tokens
  (`<<LZ>>`, `<<SYSADMIN>>`, `<<PLATFORM_TEAM>>`), named `<ticket>__<seq>.<layer>_<label>__data.SQL`.
- **Delete capture** (`capture_deletes=True`) forces bronze to `merge` with `{{ this }}` +
  `IS_DELETED`, and propagates `IS_DELETED` into the silver model + both DDLs. Needs primary keys.
- **bronze_db/schema** + **silver_db/schema** default to the landing location if omitted.
- **Config** entries nest under `extraction`; `config_prefix` (default `InsertOverwrite`) names the file.